# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

<VSCode.Cell id="#VSC-7a6bd61b" language="markdown">
# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template (filled).** This notebook compares four prompting strategies for extracting structured fields from job snippets. It runs the strategies, collects parse/accuracy/judge/cost metrics, and builds a comparison table.

Read `learner/MP1_Brief.md` for project requirements.

---

## Setup

In [ ]:
import asyncio
import json
import os
import re
import time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()   # optional: populate os.environ from .env
import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — example rates
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

## Step 1 — Load the data

In [ ]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

We implement four strategies: zero-shot, few-shot, structured (system persona + schema), and chain-of-thought (reasoning then answer).

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [{
        'role': 'user',
        'content': (
            "Extract these fields from the job posting snippet and return ONLY a JSON object with keys:\n"
            "company, role, years_experience_required (integer or null).\n\nSnippet:\n" + snippet_text
        )
    }]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2 worked examples in the prompt."""
    examples = [
        {
            'snippet': 'Acme Corp is hiring a Senior Software Engineer — 5+ years experience required.',
            'output': {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5}
        },
        {
            'snippet': 'Northwind Ltd: Data Analyst (2 years experience preferred).',
            'output': {'company': 'Northwind Ltd', 'role': 'Data Analyst', 'years_experience_required': 2}
        }
    ]
    example_text = "\n\n".join([f"Snippet: {e['snippet']}\nOutput: {json.dumps(e['output'])}" for e in examples])
    return [{
        'role': 'user',
        'content': (
            "You are an extractor. Follow the examples exactly and return only a JSON object with keys:"
            " company, role, years_experience_required.\n\n" + example_text + "\n\nNow process:\n" + snippet_text
        )
    }]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    system = {
        'role': 'system',
        'content': (
            "You are an expert recruiter and a strict JSON formatter.\n"
            "Output MUST be a single JSON object with EXACT keys: company, role, years_experience_required.\n"
            "- company: string or null\n- role: string or null\n- years_experience_required: integer or null\n"
        )
    }
    user = {
        'role': 'user',
        'content': "Extract the schema values from the snippet below and return only the JSON object.\n\nSnippet:\n" + snippet_text
    }
    return [system, user]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            'role': 'user',
            'content': (
                "Read the snippet and THINK STEP BY STEP about where the company, role, and minimum years are stated.\n"
                "After your reasoning, output a final JSON object with keys: company, role, years_experience_required.\n\nSnippet:\n" + snippet_text
            )
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

In [4]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    """
    if not text or not isinstance(text, str):
        return None
    # strip markdown/json fences
    text = re.sub(r"```\w*", "", text)
    # find first JSON-looking object
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return None
    jtxt = m.group(0)
    try:
        obj = json.loads(jtxt)
    except Exception:
        try:
            obj = json.loads(jtxt.replace("'", '"'))
        except Exception:
            return None
    # normalize keys to expected names
    out = {}
    out['company'] = obj.get('company') or obj.get('Company') or obj.get('employer')
    out['role'] = obj.get('role') or obj.get('title') or obj.get('job_title')
    yrs = obj.get('years_experience_required') or obj.get('years') or obj.get('years_experience')
    if isinstance(yrs, str):
        yrs = yrs.strip()
        if yrs.isdigit():
            yrs = int(yrs)
        else:
            m2 = re.search(r"(\d+)", yrs)
            yrs = int(m2.group(1)) if m2 else None
    out['years_experience_required'] = yrs if yrs is not None else None
    return out


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    messages = STRATEGIES[strategy_name](snippet.get('text', snippet.get('snippet', '')))
    t0 = time.perf_counter()
    resp = await client.chat.completions.create(model=MODEL, messages=messages, temperature=TEMPERATURE)
    latency = time.perf_counter() - t0
    # extract content
    try:
        content = resp.choices[0].message.content
    except Exception:
        content = str(resp)
    parsed = parse_response(content)
    # usage tokens
    usage = getattr(resp, 'usage', {}) or (resp.get('usage') if isinstance(resp, dict) else {})
    prompt_tokens = usage.get('prompt_tokens', usage.get('input_tokens', 0))
    completion_tokens = usage.get('completion_tokens', usage.get('output_tokens', 0))
    total_tokens = usage.get('total_tokens', prompt_tokens + completion_tokens)
    # cost compute
    rates = RATES.get(MODEL, {'in': 0.0, 'out': 0.0})
    cost = prompt_tokens * rates['in'] + completion_tokens * rates['out']
    return {
        'strategy': strategy_name,
        'snippet_id': snippet.get('id'),
        'raw': content,
        'parsed': parsed,
        'cost_usd': cost,
        'latency_s': latency,
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'total_tokens': total_tokens,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = []
    for sname in STRATEGIES.keys():
        for snip in snippets:
            tasks.append(run_one(sname, snip))
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from `gpt-4o` as judge

In [ ]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if extracted is None:
        return 0
    score = 0
    def norm(s):
        return (s or '').strip().lower() if isinstance(s, str) else s
    if norm(extracted.get('company')) and norm(gold.get('company')) and norm(extracted.get('company')) == norm(gold.get('company')):
        score += 1
    if norm(extracted.get('role')) and norm(gold.get('role')) and norm(extracted.get('role')) == norm(gold.get('role')):
        score += 1
    ev = extracted.get('years_experience_required')
    gv = gold.get('years_experience_required')
    try:
        if ev is None and gv is None:
            pass
        elif ev is not None and gv is not None and int(ev) == int(gv):
            score += 1
    except Exception:
        pass
    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use `gpt-4o` as a judge. Return integer 1-4.

    Rubric:
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    if extracted is None:
        return 1
    system = "You are an objective evaluator. Return a single integer 1-4 following the rubric exactly. Reply with just the integer."
    prompt = (
        "Gold JSON:\n" + json.dumps(gold, ensure_ascii=False) + "\n\n"
        "Candidate JSON:\n" + json.dumps(extracted, ensure_ascii=False) + "\n\n"
        "Rubric:\n4 — all three fields correct\n3 — two of three correct, no fabricated data\n2 — one of three correct, or fabricated a field\n1 — none correct or unparsable\n\nReply with a single integer (1-4)."
    )
    try:
        resp = await client.chat.completions.create(model=JUDGE_MODEL, messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}], temperature=0.0)
        txt = resp.choices[0].message.content.strip()
        m = re.search(r"([1-4])", txt)
        if m:
            return int(m.group(1))
    except Exception:
        pass
    return 1

In [ ]:
# Apply scoring to all 40 results
scored = []   # list of result dicts with scoring fields added
judge_tasks = []
for r in results:
    gid = r['snippet_id']
    gold_entry = golden.get(gid)
    acc = score_accuracy(r.get('parsed'), gold_entry)
    parse_success = 1 if r.get('parsed') is not None else 0
    row = dict(r)  # copy
    row['accuracy'] = acc
    row['parse_success'] = parse_success
    # queue judge call
    snippet_text = (gold_entry.get('snippet') if gold_entry else '') or next((s['snippet'] for s in snippets if s['id'] == gid), '')
    judge_tasks.append(score_llm_judge(snippet_text, r.get('parsed'), gold_entry))
    scored.append(row)

# run all judge tasks concurrently and attach scores
judge_scores = await asyncio.gather(*judge_tasks)
for row, jscore in zip(scored, judge_scores):
    row['llm_judge_score'] = int(jscore)

print(f'Scored {len(scored)} results.')

## Step 5 — Build the comparison table

In [ ]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief. Commit results when ready.

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```